# Silver Transformations

##Silver Schema creation 

In [0]:
# spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema_name}""")

In [0]:
# spark.sql(f"SHOW SCHEMAS IN {catalog_name}").display()

In [0]:
spark.sql("SHOW TABLES IN CDL.BRONZE").display()

## Configuration section

In [0]:
dbutils.widgets.text("layer","")
layer = dbutils.widgets.get("layer")
print(f"layer is {layer}")

In [0]:
catalog_name = "cdl"

bronze_schema_name = "bronze"
silver_schema_name = "silver"

storage_account_name = "datalakedevesh"
destination_container_name = "destination-cdl"

silver_base_path = (
    f"abfss://{destination_container_name}@{storage_account_name}.dfs.core.windows.net/silver/"
)

## Entity level config

In [0]:
silver_entity_config = {
    "hcp_master": {
        "bronze_table": "cdl.bronze.hcp_master_bronze",
        "silver_table": "cdl.silver.hcp_master_silver",
        "silver_path": f"{silver_base_path}hcp_master_silver/",
        "primary_keys": ["hcp_id"],
        "dedup_order_columns": ["effective_date","ingestion_file_date","bronze_load_ts"],
        "hash_columns":[
            "hcp_id",
            "hcp_name",
            "specialty",
            "city",
            "state",
            "territory_id",
            "is_active",
            "effective_date" 
        ]
    },

 "product_master": {
        "bronze_table": "cdl.bronze.product_master_bronze",
        "silver_table": "cdl.silver.product_master_silver",
        "silver_path": f"{silver_base_path}product_master_silver/",
        "primary_keys": ["product_id"],
        "dedup_order_columns": ["effective_date", "ingestion_file_date", "bronze_load_ts"],
        "hash_columns": [
            "product_id",
            "product_name",
            "brand",
            "therapy_area",
            "launch_year",
            "is_active",
            "effective_date"
        ]
    },

    "territory_master": {
        "bronze_table": "cdl.bronze.territory_master_bronze",
        "silver_table": "cdl.silver.territory_master_silver",
        "silver_path": f"{silver_base_path}territory_master_silver/",
        "primary_keys": ["territory_id"],
        "dedup_order_columns": ["effective_date", "ingestion_file_date", "bronze_load_ts"],
        "hash_columns": [
            "territory_id",
            "territory_name",
            "region",
            "zone",
            "sales_rep_id",
            "sales_rep_name",
            "effective_date"
        ]
    },
    "call_activity": {
        "bronze_table": "cdl.bronze.call_activity_bronze",
        "silver_table": "cdl.silver.call_activity_silver",
        "silver_path": f"{silver_base_path}call_activity_silver/",
        "primary_keys": ["call_id"],
        "dedup_order_columns": ["call_date", "ingestion_file_date", "bronze_load_ts"],
        "hash_columns": [
            "call_id",
            "call_date",
            "call_status",
            "call_type",
            "duration_minutes",
            "hcp_id",
            "product_id",
            "territory_id"
        ]
    },
    "rx_transactions": {
        "bronze_table": "cdl.bronze.rx_transactions_bronze",
        "silver_table": "cdl.silver.rx_transactions_silver",
        "silver_path": f"{silver_base_path}rx_transactions_silver/",
        "primary_keys": ["rx_id"],
        "dedup_order_columns": ["rx_date", "ingestion_file_date", "bronze_load_ts"],
        "hash_columns": [
            "rx_id",
            "rx_date",
            "hcp_id",
            "product_id",
            "territory_id",
            "prescription_count"
        ]
    }
}

##Package Imports

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap,
    to_date,
    current_timestamp,
    sha2,
    concat_ws
)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

##Entity level cleaning functions

In [0]:
### HCP Master
def clean_hcp_master(df):
    cleaned_df = (
        df.select(
            trim(col("hcp_id")).alias("hcp_id"),
            initcap(trim(col("hcp_name"))).alias("hcp_name"),
            initcap(trim(col("specialty"))).alias("specialty"),
            initcap(trim(col("city"))).alias("city"),
            upper(trim(col("state"))).alias("state"),
            trim(col("territory_id")).alias("territory_id"),
            col("is_active"),
            to_date(col("effective_date")).alias("effective_date"),
            to_date(col("source_file_date")).alias("source_file_date"),

            col("bronze_load_ts"),
            col("source_file_name"),
            col("source_file_path"),
            col("ingestion_file_date"),
            col("ingestion_batch_id")
        )
        .filter(col("hcp_id").isNotNull())
        .filter(col("territory_id").isNotNull())
        .filter(col("is_active") == True)
        .withColumn("silver_load_ts", current_timestamp())
    )

    return cleaned_df

In [0]:
### Product Master
def clean_product_master(df):
    cleaned_df = (
        df.select(
            trim(col("product_id")).alias("product_id"),
            initcap(trim(col("product_name"))).alias("product_name"),
            initcap(trim(col("brand"))).alias("brand"),
            initcap(trim(col("therapy_area"))).alias("therapy_area"),
            col("launch_year").cast("long").alias("launch_year"),
            col("is_active"),
            to_date(col("effective_date")).alias("effective_date"),
            to_date(col("source_file_date")).alias("source_file_date"),

            col("bronze_load_ts"),
            col("source_file_name"),
            col("source_file_path"),
            col("ingestion_file_date"),
            col("ingestion_batch_id")
        )
        .filter(col("product_id").isNotNull())
        .filter(col("is_active") == True)
        .withColumn("silver_load_ts", current_timestamp())
    )

    return cleaned_df


In [0]:
### Territory Master

def clean_territory_master(df):
    cleaned_df = (
        df.select(
            trim(col("territory_id")).alias("territory_id"),
            initcap(trim(col("territory_name"))).alias("territory_name"),
            initcap(trim(col("region"))).alias("region"),
            initcap(trim(col("zone"))).alias("zone"),
            trim(col("sales_rep_id")).alias("sales_rep_id"),
            initcap(trim(col("sales_rep_name"))).alias("sales_rep_name"),
            to_date(col("effective_date")).alias("effective_date"),
            to_date(col("source_file_date")).alias("source_file_date"),

            col("bronze_load_ts"),
            col("source_file_name"),
            col("source_file_path"),
            col("ingestion_file_date"),
            col("ingestion_batch_id")
        )
        .filter(col("territory_id").isNotNull())
        .withColumn("silver_load_ts", current_timestamp())
    )

    return cleaned_df

In [0]:
### Call Activity
def clean_call_activity(df):
    cleaned_df = (
        df.select(
            trim(col("call_id")).alias("call_id"),
            to_date(col("call_date")).alias("call_date"),
            upper(trim(col("call_status"))).alias("call_status"),
            initcap(trim(col("call_type"))).alias("call_type"),
            col("duration_minutes").cast("long").alias("duration_minutes"),
            trim(col("hcp_id")).alias("hcp_id"),
            trim(col("product_id")).alias("product_id"),
            trim(col("territory_id")).alias("territory_id"),
            to_date(col("source_file_date")).alias("source_file_date"),
            upper(trim(col("source_system"))).alias("source_system"),

            col("bronze_load_ts"),
            col("source_file_name"),
            col("source_file_path"),
            col("ingestion_file_date"),
            col("ingestion_batch_id")
        )
        .filter(col("call_id").isNotNull())
        .filter(col("hcp_id").isNotNull())
        .filter(col("product_id").isNotNull())
        .filter(col("territory_id").isNotNull())
        .filter(col("call_date").isNotNull())
        .withColumn("silver_load_ts", current_timestamp())
    )

    return cleaned_df

In [0]:
### Rx Transactions
def clean_rx_transactions(df):
    cleaned_df = (
        df.select(
            trim(col("rx_id")).alias("rx_id"),
            to_date(col("rx_date")).alias("rx_date"),
            trim(col("hcp_id")).alias("hcp_id"),
            trim(col("product_id")).alias("product_id"),
            trim(col("territory_id")).alias("territory_id"),
            col("prescription_count").cast("long").alias("prescription_count"),
            to_date(col("source_file_date")).alias("source_file_date"),
            upper(trim(col("source_system"))).alias("source_system"),

            col("bronze_load_ts"),
            col("source_file_name"),
            col("source_file_path"),
            col("ingestion_file_date"),
            col("ingestion_batch_id")
        )
        .filter(col("rx_id").isNotNull())
        .filter(col("hcp_id").isNotNull())
        .filter(col("product_id").isNotNull())
        .filter(col("territory_id").isNotNull())
        .filter(col("rx_date").isNotNull())
        .filter(col("prescription_count") >= 0)
        .withColumn("silver_load_ts", current_timestamp())
    )

    return cleaned_df


## Cleaning function router

In [0]:
def clean_silver_entity(entity_name, df):
    if entity_name == "hcp_master":
        return clean_hcp_master(df)

    elif entity_name == "product_master":
        return clean_product_master(df)

    elif entity_name == "territory_master":
        return clean_territory_master(df)

    elif entity_name == "call_activity":
        return clean_call_activity(df)

    elif entity_name == "rx_transactions":
        return clean_rx_transactions(df)

    else:
        raise ValueError(f"No cleaning function defined for entity: {entity_name}")

## Deduplication function

In [0]:
def deduplicate_latest_record(df, primary_keys, order_columns):
    window_spec = (
        Window
            .partitionBy(*primary_keys)
            .orderBy(*[col(c).desc() for c in order_columns])
    )

    dedup_df = (
        df.withColumn("row_num", row_number().over(window_spec))
          .filter(col("row_num") == 1)
          .drop("row_num")
    )

    return dedup_df

## Hash function

In [0]:
def add_record_hash(df, hash_columns):
    hash_expr_columns = [
        col(c).cast("string")
        for c in hash_columns
    ]

    hashed_df = (
        df.withColumn(
            "record_hash",
            sha2(concat_ws("||", *hash_expr_columns), 256)
        )
    )

    return hashed_df

## Write function

In [0]:
def write_to_silver(df, silver_table, silver_path, write_mode="overwrite"):
    print(f"Writing Silver table: {silver_table}")
    print(f"Target path: {silver_path}")

    (
        df.write
            .format("delta")
            .mode(write_mode)
            .option("path", silver_path)
            .option("overwriteSchema", "true")
            .saveAsTable(silver_table)
    )

    print(f"Successfully written: {silver_table}")

## Caller function

In [0]:
def process_silver_entity(entity_name):
    config = silver_entity_config[entity_name]

    bronze_table = config["bronze_table"]
    silver_table = config["silver_table"]
    silver_path = config["silver_path"]
    primary_keys = config["primary_keys"]
    dedup_order_columns = config["dedup_order_columns"]
    hash_columns = config["hash_columns"]

    print(f"Started Silver processing for: {entity_name}")
    print(f"Reading Bronze table: {bronze_table}")

    bronze_df = spark.table(bronze_table)

    cleaned_df = clean_silver_entity(entity_name, bronze_df)

    dedup_df = deduplicate_latest_record(
        cleaned_df,
        primary_keys,
        dedup_order_columns
    )

    final_df = add_record_hash(
        dedup_df,
        hash_columns
    )

    write_to_silver(
        final_df,
        silver_table,
        silver_path,
        write_mode="overwrite"
    )

    print(f"Completed Silver processing for: {entity_name}")

    return final_df

# print(process_silver_entity("hcp_master"))

## Runner function

In [0]:
def silver_transformation():

    silver_results = []

    for entity_name in silver_entity_config.keys():
        final_df = process_silver_entity(entity_name)
        silver_results.append({
            "entity_name": f"{entity_name}_silver",
            "silver_table": silver_entity_config[entity_name]["silver_table"],
            "silver_path": silver_entity_config[entity_name]["silver_path"],
            "row_count": final_df.count()
            })
    
    return silver_results

silver_transformation()

In [0]:
# spark.sql(f"describe detail cdl.silver.product_master_silver").select("name","location","format").display()